# EDA một trang: 1 115 cửa hàng Rossmann khác nhau thế nào?

**Bối cảnh.** Head of Operations mới của chuỗi Rossmann nhắn: *"Tôi vừa nhận 1 115 cửa hàng. Cho tôi biết chúng
khác nhau thế nào về doanh số, và những cửa hàng doanh số thấp có điểm gì chung. Tôi cần biết nên bắt đầu từ đâu,
gửi tôi một trang trước thứ Sáu."*

Dữ liệu: bộ `store-sales` (`train.csv`: doanh số ngày 01/01/2013–31/07/2015, một dòng mỗi cửa hàng-ngày kể cả
ngày đóng cửa; `store.csv`: `StoreType`, `Assortment`, `CompetitionDistance`, `Promo2`, ...). Đọc thêm phần 7 lý
thuyết (quy trình EDA có câu hỏi, sản phẩm một trang).

Việc của bạn:

1. Viết `store_profile(train, stores)`: bảng một dòng mỗi cửa hàng, nền cho mọi phân tích sau (được chấm).
2. EDA tự do trong các cell `explore` (có khung gợi ý bên dưới, thêm cell tuỳ ý), time-box khoảng 3 giờ.
3. Viết `starter/REPORT.md` theo khung có sẵn: một trang (250-700 từ), headline có con số, 3 phát hiện.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) bị bỏ qua khi chấm.
- Chấm phần tự động: `uv run pytest tracks/data-analyst/modules/07-descriptive-eda/exercises/advanced/01-store-eda-one-pager -q`. Nội dung báo cáo chấm theo rubric trong `SOLUTION.md`.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

STORE = ensure("store-sales")
train = pd.read_csv(STORE / "train.csv", dtype={"StateHoliday": str}, parse_dates=["Date"])
stores = pd.read_csv(STORE / "store.csv")
print(train.shape, stores.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Hồ sơ cửa hàng

`store_profile(train, stores) -> DataFrame`. Chỉ dùng ngày bán (`Open == 1` và `Sales > 0`). Index là `Store` (sắp
tăng, chỉ cửa hàng có ít nhất một ngày bán), các cột đúng thứ tự:

| Cột | Định nghĩa |
| --- | --- |
| `open_days` | số ngày bán |
| `mean_sales`, `median_sales` | mean, median doanh số ngày bán |
| `cv_sales` | SD mẫu / mean của doanh số ngày bán |
| `sales_per_customer` | tổng doanh số / tổng số khách (ngày bán) |
| `promo_lift` | mean doanh số ngày `Promo == 1` chia mean ngày `Promo == 0`, trừ 1; NaN nếu thiếu một trong hai loại ngày |
| `StoreType`, `Assortment`, `CompetitionDistance` | lấy từ `stores` theo `Store` (NaN nếu không có) |

In [ ]:
def store_profile(train: pd.DataFrame, stores: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi cửa hàng có ngày bán: open_days, mean_sales, median_sales, cv_sales,
    sales_per_customer, promo_lift, StoreType, Assortment, CompetitionDistance."""
    # TODO
    raise NotImplementedError

## 2. EDA theo câu hỏi

Khung gợi ý (đổi, bỏ, thêm tuỳ ý; mỗi cell nên mở đầu bằng câu hỏi nó trả lời):

1. **Phân phối**: doanh số TB mỗi cửa hàng phân tán thế nào (center, spread, shape)? Cửa hàng lớn nhất gấp bao
   nhiêu lần nhỏ nhất, p90 gấp bao nhiêu p10?
2. **Định nghĩa "doanh số thấp"**: nhóm nào (quartile dưới? dưới median bao nhiêu?), bao nhiêu cửa hàng?
3. **Điểm chung**: so nhóm thấp với các nhóm khác theo `StoreType`, `Assortment`, `CompetitionDistance`, `Promo2`,
   `sales_per_customer`, `promo_lift`, `cv_sales`. So **tỉ lệ** trong nhóm, không chỉ số đếm.
4. **Kiểm bẫy**: có cửa hàng thiếu dữ liệu một giai đoạn không (phần 5 lý thuyết)? Doanh số thấp vì ít khách hay vì
   mỗi khách chi ít?

In [ ]:
# explore: hồ sơ và phân phối doanh số TB mỗi cửa hàng
profile = store_profile(train, stores)
print(profile.describe().round(2))
s = profile.mean_sales
print(f"skew {s.skew():.2f}; max/min {s.max() / s.min():.1f}; p90/p10 {s.quantile(0.9) / s.quantile(0.1):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(s, bins=50, color="#4c78a8")
ax[0].set_title("Doanh số TB/ngày bán của 1 115 cửa hàng", loc="left")
x = np.sort(s)
ax[1].plot(x, np.arange(1, len(x) + 1) / len(x))
ax[1].set_title("ECDF", loc="left")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# explore: so quartile doanh số theo đặc điểm cửa hàng (ví dụ; thêm cột Promo2 từ stores nếu cần)
quartile = pd.qcut(profile.mean_sales, 4, labels=["Q1 thấp", "Q2", "Q3", "Q4 cao"])
print(profile.groupby(quartile, observed=True)[["sales_per_customer", "promo_lift", "cv_sales", "CompetitionDistance"]].median().round(3))
print(pd.crosstab(quartile, profile.StoreType, normalize="index").round(3))

In [ ]:
# explore: chỗ của bạn

## 3. Viết báo cáo

Mở `starter/REPORT.md`, điền theo khung. Một trang: người đọc không cần mở notebook, không cần biết tên cột. Mỗi
phát hiện có một con số, một tham chiếu để so, và tên chart/bảng trong notebook làm bằng chứng.